In [3]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as ss
import sklearn
import os
import sktree

In [4]:
# load data
wise = pd.read_csv('./wise.matrix')
wise_all = pd.read_csv('./wise_All.matrix')
display(wise.head())
print(wise.shape)
wise_y = wise['Cancer Status']
wise_X = wise.drop(columns = ['Cancer Status', 'Sample', 'Tumor type', 'Stage'])
print(wise_X.shape)

,Sample,Cancer Status,Stage,Tumor type,1:1-10000000,1:10000001-20000000,1:20000001-30000000,1:30000001-40000000,1:40000001-50000000,1:50000001-60000000,...,21:10000001-20000000,21:20000001-30000000,21:30000001-40000000,21:40000001-50000000,22:1-10000000,22:10000001-20000000,22:20000001-30000000,22:30000001-40000000,22:40000001-50000000,22:50000001-60000000
0,S0035.INDI_998_PLS_1,1,IV,Ovary,0.107617,-0.089664,0.088308,0.161890,0.195141,0.059390,...,-1.072653,-0.931427,-0.835283,-0.817325,0.0,-1.819371,-1.749369,-1.684808,-1.647312,-1.532646
1,S0035.INDI_983_PLS_1,1,IV,Ovary,-0.772378,-1.087627,-0.949005,0.489450,-0.670958,-0.221170,...,-0.897900,-0.524623,-0.922448,-0.793396,0.0,-0.675361,-0.943580,-0.758821,-0.932769,-0.749762
2,S0035.INDI_510_PLS_1A,1,IV,Ovary,-1.519912,-1.795528,-1.847208,0.015104,0.541656,0.411293,...,-0.796892,0.147432,-0.221206,-0.270937,0.0,-3.158424,-1.795942,-1.927368,-1.657170,-1.819905
3,S0063.INDIA_3343_PLS_1,1,IIIA,Lung,3.058270,2.851708,2.665363,-0.828755,-0.752899,-0.695762,...,-1.603144,-1.409415,-0.781825,-0.207052,0.0,2.169607,1.920353,-0.745165,-0.920590,-0.716167
4,S0064.INDIA_3360_PLS_1,1,IIB,Lung,0.017034,0.189540,-0.901394,-0.537247,-0.986964,-0.564267,...,0.044192,0.188299,0.263522,1.150110,0.0,-0.045028,-0.220047,0.243085,1.025034,1.885327


(704, 304)
(704, 300)


In [5]:
display(wise_all.head())
print(wise_all.shape)
wise_all_y = wise_all['Cancer Status']
wise_all_X = wise_all.drop(columns = ['Run','Library','Cancer Status', 'Sample', 'Tumor type', 'Stage'])
print(wise_all_X.shape)

,Run,Sample,Library,Cancer Status,Stage,Tumor type,1:1-10000000,1:10000001-20000000,1:20000001-30000000,1:30000001-40000000,...,21:10000001-20000000,21:20000001-30000000,21:30000001-40000000,21:40000001-50000000,22:1-10000000,22:10000001-20000000,22:20000001-30000000,22:30000001-40000000,22:40000001-50000000,22:50000001-60000000
0,S0028,INDI_918_PLS_1A,A387-12,Cancer,IV,Stomach,0.468038,-3.145530,-1.712623,-0.735046,...,-1.135428,0.910290,-0.814004,0.465840,NaN,-2.455095,0.242265,-0.999295,0.586578,4.020335
1,S0028,INDI_980_PLS_1,A387-13,Cancer,IV,Stomach,0.632628,-3.349720,-0.246645,-0.336667,...,-0.728640,0.982447,-1.233654,1.355670,NaN,2.373436,4.932168,-0.335874,0.462123,2.862172
2,S0034,INDI_580_PLS_1A,A396-13,Cancer,IV,Colorectal,-2.123124,-2.840198,-2.896575,-2.638872,...,0.540503,-0.127143,-0.174641,0.031823,NaN,0.298749,-0.050700,-0.240639,0.109563,0.532129
3,S0034,INDI_730_PLS_1A,A396-04,Cancer,IV,Pancreas,-1.318004,-1.953182,-1.099582,-0.309286,...,0.567452,-0.766169,-0.743883,1.192330,NaN,-2.766405,0.101859,-1.526696,0.981383,-1.887319
4,S0034,INDI_481_PLS_1A,A396-10,Cancer,IV,Liver,0.199915,-1.960120,-1.531292,-0.322800,...,-2.222321,0.856329,-0.179628,-0.078619,NaN,-7.305575,-3.274674,-1.136304,-0.444130,0.926091


(1991, 306)
(1991, 300)


In [6]:
# train a Random Forest model on the WISE data
from sklearn.ensemble import RandomForestClassifier
from joblib import Parallel, delayed
from scipy.stats import false_discovery_control
from statsmodels.stats.multitest import multipletests

def statistics(feature_importance, idx, n_estimators=100):
    r"""
    Helper function that calulates the feature importance
    test statistic.
    """
    stat = np.zeros(len(feature_importance[0]))
    for ii in range(n_estimators):
        r = ss.rankdata(1 - feature_importance[idx[ii]], method="max")
        r_0 = ss.rankdata(1 - feature_importance[idx[n_estimators + ii]], method="max")
        stat += (r_0 > r) * 1
    stat /= n_estimators
    return stat


def perm_stat(feature_importance, n_estimators=1000):
    r"""
    Helper function that calulates the null distribution.
    """
    idx = list(range(2 * n_estimators))
    # print(idx)
    np.random.shuffle(idx)
    # print(idx)
    return statistics(feature_importance, idx)


def test(feature_importance, n_repeats=1000, n_jobs=-1, n_estimators=1000, method="holm"):
    r"""
    Calculates p values for fearture imprtance test.
    Parameters
    ----------
    X : ArrayLike of shape (n_samples, n_features)
        The data matrix.
    y : ArrayLike of shape (n_samples, n_outputs)
        The target matrix.
    n_repeats : int, optional
        Number of times to sample the null distribution, by default 1000.
    n_jobs : int, optional
        Number of workers to use, by default 1000.
    Returns
    -------
    stat : float
        The computed discriminability statistic.
    pvalue : float
        The computed one sample test p-value.
    """
    stat = statistics(feature_importance, list(range(2 * n_estimators)), n_estimators=n_estimators)
    # print(stat)
    null_stat = Parallel(n_jobs=n_jobs)(
        delayed(perm_stat)(feature_importance, n_estimators=n_estimators)
        for _ in range(n_repeats)
    )
    count = np.sum((null_stat >= stat) * 1, axis=0)
    p_val = (1 + count) / (1 + n_repeats)
    # ps_adusted = false_discovery_control(p_val, method=method)
    ps_adusted = multipletests(p_val, method=method)[1]
    return stat, p_val,ps_adusted

In [29]:

X = wise_X
y = wise_y
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
# train using 
model = RandomForestClassifier(n_estimators=1000, n_jobs=10)
permuted_model = RandomForestClassifier(n_estimators=1000, n_jobs=10)
model.fit(X_train,y_train)
# test the model on the test data
y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

# feature_importance_origin = model.feature_importances_
# del model
feature_importance_origin = [tree.feature_importances_ for tree in model.estimators_]
# print(len(feature_importance_origin[0]))
y_cp = y_train.copy()
# shuffle the labels
y_cp = np.random.permutation(y_cp)
# np.random.shuffle(y_cp)
permuted_model.fit(X_train,y_cp)
permuted_feature_importance = [tree.feature_importances_ for tree in permuted_model.estimators_]
# del permuted_model
feature_importance = np.concatenate((feature_importance_origin,permuted_feature_importance))
# print(len(feature_importance))
stat, p_val, ps_adusted = test(feature_importance,
            n_repeats = 1000,
            n_jobs = 10,
            n_estimators = 1000)
# print(p_val)


0.9716312056737588


In [30]:
# sort the columns by p-value
p_val_df = pd.DataFrame(p_val, columns = ['p_value'])
p_val_df['feature'] = X_train.columns
p_val_sorted = p_val_df.sort_values(by = 'p_value')
k = len(p_val_sorted)
rev_idx = np.linspace(1, k, k)[::-1]
# multiply the p-values by the number of tests
p_val_sorted['p_value_adjusted'] = p_val_sorted['p_value'] * rev_idx/k
# sort the p-values by the adjusted p-values
p_val_sorted.sort_values(by = 'p_value_adjusted')[:].drop(columns = ['p_value']).to_csv('wise_feature_importance.csv', index = False)


In [59]:
p_val_sorted


,p_value,feature,p_value_adjusted
156,0.000999,8:130000001-140000000,0.000999
157,0.000999,8:140000001-150000000,0.000996
155,0.005994,8:120000001-130000000,0.005954
154,0.007992,8:110000001-120000000,0.007912
288,0.015984,20:60000001-70000000,0.015771
...,...,...,...
181,1.000000,10:80000001-90000000,0.016667
236,1.000000,14:90000001-100000000,0.013333
189,1.000000,11:20000001-30000000,0.010000
13,1.000000,1:130000001-140000000,0.006667


In [58]:
p_val_sorted

,p_value,feature,p_value_adjusted
156,0.000999,8:130000001-140000000,0.000999
157,0.000999,8:140000001-150000000,0.000996
155,0.005994,8:120000001-130000000,0.005954
154,0.007992,8:110000001-120000000,0.007912
288,0.015984,20:60000001-70000000,0.015771
...,...,...,...
181,1.000000,10:80000001-90000000,0.016667
236,1.000000,14:90000001-100000000,0.013333
189,1.000000,11:20000001-30000000,0.010000
13,1.000000,1:130000001-140000000,0.006667


In [32]:
from sklearn.inspection import permutation_importance
result = permutation_importance(model, X_test, y_test, n_repeats=10, n_jobs=-5)
sorted_importances_idx = result.importances_mean.argsort()
# sort the columns by sorted_idx
importances = pd.DataFrame(
    result.importances[sorted_importances_idx].T,
    columns=X_train.columns[sorted_importances_idx]
)
display(importances.head(20))

,4:180000001-190000000,4:120000001-130000000,8:1-10000000,4:80000001-90000000,4:130000001-140000000,5:180000001-190000000,4:100000001-110000000,8:20000001-30000000,4:110000001-120000000,17:1-10000000,...,5:130000001-140000000,5:120000001-130000000,5:110000001-120000000,5:100000001-110000000,5:90000001-100000000,5:150000001-160000000,8:70000001-80000000,8:140000001-150000000,8:90000001-100000000,8:130000001-140000000
0,-0.007092,-0.007092,-0.007092,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.007092,0.007092,0.000000
1,0.000000,-0.007092,0.000000,0.000000,0.000000,0.000000,-0.007092,0.000000,-0.007092,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.007092,0.007092,0.007092,0.007092
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.007092,0.007092,0.007092,0.014184
3,-0.007092,0.000000,-0.007092,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.007092,0.007092,0.007092,0.014184
4,-0.007092,0.000000,0.000000,-0.007092,-0.007092,-0.007092,0.000000,0.000000,-0.007092,-0.007092,...,0.0,0.0,0.0,0.0,0.0,0.0,0.007092,0.007092,0.007092,0.014184
5,-0.007092,0.000000,-0.007092,0.000000,0.000000,-0.007092,0.000000,-0.007092,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.007092,0.000000,0.007092,0.014184
6,-0.007092,0.000000,-0.007092,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.007092,0.007092,0.007092
7,-0.007092,-0.007092,0.000000,0.000000,0.000000,0.000000,-0.007092,-0.007092,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.007092,0.007092
9,-0.007092,-0.007092,0.000000,-0.007092,-0.007092,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.007092,0.007092,0.007092,0.014184


In [40]:
pd.DataFrame(wise_X.columns[np.argsort(ss.rankdata(p_val, method='min'))[:20]])

,0
0,8:130000001-140000000
1,8:140000001-150000000
2,8:120000001-130000000
3,8:110000001-120000000
4,20:60000001-70000000
5,8:90000001-100000000
6,20:50000001-60000000
7,4:120000001-130000000
8,1:170000001-180000000
9,8:70000001-80000000


In [41]:
# get the mean feature importance from the random forest model
feature_importance_origin = [tree.feature_importances_ for tree in model.estimators_]
feature_importance = np.mean(feature_importance_origin, axis=0)

print(feature_importance.shape)
# get the column names
columns = wise_X.columns
print(columns.shape)
# create a dataframe
df = pd.DataFrame({'feature': columns, 'importance': feature_importance})
# sort the dataframe
df.sort_values('importance', ascending = False).to_csv("mean_feature_importance.csv",index=False)
df.sort_values('importance', ascending = False)[:20]


(300,)
(300,)


,feature,importance
156,8:130000001-140000000,0.067590
157,8:140000001-150000000,0.055604
154,8:110000001-120000000,0.042967
155,8:120000001-130000000,0.040197
150,8:70000001-80000000,0.039449
152,8:90000001-100000000,0.036767
153,8:100000001-110000000,0.022659
17,1:170000001-180000000,0.021825
21,1:210000001-220000000,0.021697
145,8:20000001-30000000,0.020939


In [17]:
# pd.DataFrame(wise_X.columns[np.argsort(ss.rankdata(ps_adusted, method='min'))[:]]).to_csv('wise_rank.csv', index=False)
# pd.DataFrame(wise_X.columns[np.argsort(ss.rankdata(p_val, method='min'))[:]]).to_csv('wise_rank_pval.csv', index=False)

In [46]:

X = wise_all_X
y = wise_all_y
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
# train using 
model = RandomForestClassifier(n_estimators=1000, n_jobs=10)
permuted_model = RandomForestClassifier(n_estimators=1000, n_jobs=10)
model.fit(X_train,y_train)
# test the model on the test data
y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

# feature_importance_origin = model.feature_importances_
# del model
feature_importance_origin = [tree.feature_importances_ for tree in model.estimators_]
# print(len(feature_importance_origin[0]))
y_cp = y_train.copy()
# shuffle the labels
y_cp = np.random.permutation(y_cp)
# np.random.shuffle(y_cp)
permuted_model.fit(X_train,y_cp)
permuted_feature_importance = [tree.feature_importances_ for tree in permuted_model.estimators_]
# del permuted_model
feature_importance = np.concatenate((feature_importance_origin,permuted_feature_importance))
# print(len(feature_importance))
stat, p_val, ps_adusted = test(feature_importance,
            n_repeats = 1000,
            n_jobs = 10,
            n_estimators = 1000)

0.8095238095238095


In [48]:
# sort the columns by p-value
p_val_df_all = pd.DataFrame(p_val, columns = ['p_value'])
p_val_df_all['feature'] = X_train.columns
p_val_sorted_all = p_val_df.sort_values(by = 'p_value')
k = len(p_val_sorted_all)
rev_idx = np.linspace(1, k, k)[::-1]
# multiply the p-values by the number of tests
p_val_sorted_all['p_value_adjusted'] = p_val_sorted_all['p_value'] * rev_idx/k
# sort the p-values by the adjusted p-values
p_val_sorted_all.sort_values(by = 'p_value_adjusted')[:].drop(columns = ['p_value']).to_csv('wise_feature_importance_all.csv', index = False)
p_val_sorted_all.sort_values(by = 'p_value_adjusted')[:].drop(columns = ['p_value'])[:20]

,feature,p_value_adjusted
157,8:140000001-150000000,0.000996
156,8:130000001-140000000,0.000999
258,16:90000001-100000000,0.003333
155,8:120000001-130000000,0.005954
13,1:130000001-140000000,0.006667
154,8:110000001-120000000,0.007912
189,11:20000001-30000000,0.010000
236,14:90000001-100000000,0.013333
288,20:60000001-70000000,0.015771
181,10:80000001-90000000,0.016667


In [49]:
# get the mean feature importance from the random forest model
feature_importance_origin = [tree.feature_importances_ for tree in model.estimators_]
feature_importance_all = np.mean(feature_importance_origin, axis=0)

print(feature_importance_all.shape)
# get the column names
columns = wise_all_X.columns
print(columns.shape)
# create a dataframe
df_all = pd.DataFrame({'feature': columns, 'importance': feature_importance_all})
# sort the dataframe
df_all.sort_values('importance', ascending = False).to_csv("mean_feature_importance_all.csv",index=False)
df_all.sort_values('importance', ascending = False)[:20]

(300,)
(300,)


,feature,importance
34,2:90000001-100000000,0.017427
133,7:60000001-70000000,0.015793
192,11:50000001-60000000,0.015028
157,8:140000001-150000000,0.014835
25,2:1-10000000,0.014142
63,3:130000001-140000000,0.013137
288,20:60000001-70000000,0.013056
287,20:50000001-60000000,0.012378
142,7:150000001-160000000,0.010871
134,7:70000001-80000000,0.010233


In [56]:
feature_importance_df_all_sorted = df_all.sort_values(by = 'importance', ascending = False)
k = len(feature_importance_df_all_sorted)
rev_idx = np.linspace(1, k, k)[::-1]
# multiply the p-values by the number of tests
feature_importance_df_all_sorted['importance_adjusted'] = feature_importance_df_all_sorted['importance'] * rev_idx/k
feature_importance_df_all_sorted.sort_values(by = 'importance_adjusted', ascending=False)[:20]

,feature,importance,importance_adjusted
34,2:90000001-100000000,0.017427,0.017427
133,7:60000001-70000000,0.015793,0.015741
192,11:50000001-60000000,0.015028,0.014928
157,8:140000001-150000000,0.014835,0.014686
25,2:1-10000000,0.014142,0.013954
63,3:130000001-140000000,0.013137,0.012919
288,20:60000001-70000000,0.013056,0.012795
287,20:50000001-60000000,0.012378,0.012089
142,7:150000001-160000000,0.010871,0.010581
134,7:70000001-80000000,0.010233,0.009926
